# 02 · First chat with an LLM (Version 1 and Version 2)

**My plan:**
1. Send my first message to an LLM
2. Measure time, tokens and cost
3. Build V1: the LLM answers customers with no tools
4. Build V2: give the LLM tools so it can really *do* things

> Put ONE key in the `.env` file: OpenAI, Anthropic or Gemini (see README).
> No key? It uses the free local model in Ollama instead (slower, less smart, but free).

In [ ]:
import sys
sys.path.append("..")   # so Python can find the budget_agent folder

import pandas as pd
pd.set_option("display.max_colwidth", 80)
from dotenv import load_dotenv
load_dotenv("../.env")

from budget_agent.llm import make_llm

llm = make_llm()    # picks the provider from the key in your .env
print("Provider:", llm.provider, "| Model:", llm.model)

## Step 1: My very first message

In [ ]:
reply = llm.chat(
    system="You are a helpful assistant.",
    history=[{"role": "user", "content": "In one sentence: what is an AI agent?"}],
)
print(reply.text)

## Step 2: How long did it take, and what did it cost?

In [ ]:
reply.stats

In [ ]:
from budget_agent import config

price_in, price_out = config.PRICES_PER_MILLION.get(llm.model, (0, 0))
print(f"Price: ${price_in} per million input tokens, ${price_out} per million output tokens")
print(f"This one message cost ${reply.stats.cost_usd:.5f} and took {reply.stats.seconds:.1f}s")
print(f"1,000 messages like it would cost about ${reply.stats.cost_usd * 1000:.2f}")

Tiny for one message. But a shop gets thousands every day, and agent messages are much longer than this one.

## Step 3: Version 1, the LLM with no tools

I put my prompt and the cost counting into `budget_agent/agent.py` so I don't repeat code.

In [ ]:
from budget_agent.agent import SupportAgent

v1 = SupportAgent(use_tools=False)
result = v1.run("where is my order NM10005?")
print(result.reply)
print(f"\n{result.seconds:.1f}s, ${result.cost_usd:.4f}")

**Problem:** V1 is polite, but it cannot see the order. It can only talk.
A real support bot must **look things up and take actions**. That's what tools are for.

## Step 4: Version 2, the LLM with tools

Tools are plain Python functions in `budget_agent/tools.py`. The LLM can't run them itself. It *asks* us, we run them, and we send back the result.

```
message → LLM → "please run lookup_order(NM10005)" → we run it → result → LLM → reply
```

In [ ]:
from budget_agent.tools import TOOL_DEFINITIONS

[tool["name"] for tool in TOOL_DEFINITIONS]

In [ ]:
v2 = SupportAgent(use_tools=True)

def show(result):
    for name, tool_input, output in result.tool_calls:
        print(f"🔧 {name}({tool_input})\n   → {output[:150]}")
    print(f"\n💬 {result.reply}")
    print(f"\n⏱  {result.seconds:.1f}s   💲 ${result.cost_usd:.4f}   LLM calls: {len(result.calls)}")

show(v2.run("where is my order NM10005?"))

Now it actually looked up the order! Let me try harder things.

In [ ]:
# A refund on a delivered order
from budget_agent import data

orders = data.load_orders()
delivered = orders.query("status == 'Delivered' and amount_inr < 10000").order_id.iloc[0]
show(v2.run(f"my order {delivered} arrived broken, I want a refund"))

In [ ]:
# An angry customer → should go to a human
show(v2.run("this is the 3rd time my delivery is late. I want to talk to a manager NOW"))

In [ ]:
# Guardrail test: refunds above ₹10,000 must NOT happen automatically
big = orders.query("status == 'Delivered' and amount_inr > 10000").order_id.iloc[0]
show(v2.run(f"refund {big}, I don't like it"))

In [ ]:
# Everything the agent changed in the shop
v2.shop.actions

## What I learned
- V1 can talk but can't act. V2 is a real agent: it looks things up, takes actions, and follows rules.
- The guardrail (big refunds need a human) is in **my code**, not in the prompt. The LLM can't skip it.
- **But:** every message uses the big LLM, often 2-3 calls. Even "what payment methods do you accept?" costs money and takes seconds.

Next: can a tiny ML model handle the easy messages? → `03_llm_labels_ml_learns.ipynb`